# Password & Security Advisor using GenAI + Agentic AI### Day X: Mini ProjectAn AI agent that analyzes password strength, checks breach exposure, and gives grounded security advice using tool-calling + a mini RAG knowledge base.

In [ ]:
# Step 1: Install all important Modules!pip install langchain!pip install langchain_community!pip install langchain-google-genai!pip install langchain-groq!pip install faiss-cpu!pip install sentence-transformers!pip install requests!pip install streamlit# Kernel Restart

In [ ]:
# Steps to be follow# 1. Load password / account input# 2. Tool: Rule-based password strength + entropy check# 3. Tool: Breach check (Have I Been Pwned, k-anonymity)# 4. Tool: Mini RAG - retrieve security guidance from knowledge base# 5. Tool: Strong password/passphrase generator# 6. Agent: combine tools using create_agent# 7. Agent: Report Insights (plain-language explanation)# 8. Report view on Streamlit Web Page# H/w: Autonomous monitoring agent (perceive->decide->act loop) for rotation reminders

In [ ]:
# Step 3: Load all modulesimport pandas as pdimport numpy as npimport hashlibimport mathimport reimport requestsimport osimport timeimport langchainimport langchain_communityfrom langchain_google_genai import ChatGoogleGenerativeAIfrom langchain_groq import ChatGroqfrom langchain.agents import create_agentfrom langchain_community.vectorstores import FAISSfrom langchain_community.embeddings import HuggingFaceEmbeddingsfrom langchain_text_splitters import RecursiveCharacterTextSplitterfrom langchain_core.documents import Documentimport streamlit as stprint("Modules Loaded Successfully!!")

In [ ]:
# Step 4: Model creation# SECURITY NOTE: never hardcode API keys directly in a notebook you might# share or push to GitHub. Use environment variables / Colab secrets instead.# In Colab: use the "Secrets" tab (key icon in left sidebar) to store these.from google.colab import userdataGOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')GROQ_API_KEY = userdata.get('GROQ_API_KEY')gemini_llm = ChatGoogleGenerativeAI(    model="gemini-2.0-flash",    google_api_key=GOOGLE_API_KEY    )groq_llm = ChatGroq(    model="llama-3.3-70b-versatile",    api_key=GROQ_API_KEY    )print("Done")

## Tool 1: Rule-based Password Strength CheckerThis runs BEFORE any LLM call. The score and entropy are computed deterministically — the agent explains these numbers, it never invents them.

In [ ]:
COMMON_PATTERNS = [r"1234", r"qwerty", r"asdf", r"password", r"letmein",                   r"admin", r"welcome", r"abc123", r"iloveyou"]KEYBOARD_WALKS = ["qwertyuiop", "asdfghjkl", "zxcvbnm", "1234567890"]def _char_set_size(password):    size = 0    if re.search(r"[a-z]", password): size += 26    if re.search(r"[A-Z]", password): size += 26    if re.search(r"[0-9]", password): size += 10    if re.search(r"[^a-zA-Z0-9]", password): size += 32    return size or 1def check_password_strength(password: str) -> str:    """Compute entropy, strength category, and specific weaknesses of a    password. Always call this before giving strength advice."""    charset = _char_set_size(password)    entropy = round(len(password) * math.log2(charset), 2)    issues = []    lower = password.lower()    for pattern in COMMON_PATTERNS:        if re.search(pattern, lower):            issues.append(f"contains common weak pattern '{pattern}'")    for walk in KEYBOARD_WALKS:        for i in range(len(walk) - 3):            if walk[i:i+4] in lower:                issues.append(f"contains keyboard-walk sequence '{walk[i:i+4]}'")                break    if len(password) < 8:        issues.append("shorter than recommended 8-character minimum")    if password.lower() == password or password.upper() == password:        issues.append("uses only one letter case")    if not re.search(r"\d", password):        issues.append("contains no digits")    if entropy < 28: category = "very weak"    elif entropy < 36: category = "weak"    elif entropy < 60: category = "reasonable"    elif entropy < 128: category = "strong"    else: category = "excellent"    base_score = min(100, round((entropy / 128) * 100))    score = max(0, base_score - min(base_score, len(issues) * 12))    issues_text = '; '.join(issues) if issues else 'none detected'    return (f"score={score}/100, entropy={entropy} bits, category={category}, "            f"length={len(password)}, issues=[{issues_text}]")# quick testprint(check_password_strength("Password123!"))print(check_password_strength("Xk9#mQ2$vL7@pR4"))

## Tool 2: Breach Checker (Have I Been Pwned)Uses HIBP's k-anonymity model — only the first 5 characters of the SHA-1 hash are ever sent over the network. The real password never leaves this notebook.

In [ ]:
def check_breach_database(password: str) -> str:    """Check whether a password has appeared in known public data breaches    via the Have I Been Pwned Pwned Passwords API. Always call this before    telling a user whether their password is compromised."""    sha1 = hashlib.sha1(password.encode("utf-8")).hexdigest().upper()    prefix, suffix = sha1[:5], sha1[5:]    try:        resp = requests.get(f"https://api.pwnedpasswords.com/range/{prefix}", timeout=5)        resp.raise_for_status()    except requests.RequestException as e:        return f"breach check failed (network error): {e}"    for line in resp.text.splitlines():        hash_suffix, count = line.split(":")        if hash_suffix == suffix:            return f"BREACHED: seen {count} times in known breach dumps"    return "not found in known breach dumps"# quick testprint(check_breach_database("password123"))

## Tool 3: Mini RAG - Security Knowledge BaseSame idea as your EDA agent's report step, but here we build a small FAISS vector store from security guideline text so the agent's advice is grounded in real guidance (NIST-style rules, common attack patterns) instead of made up on the spot.

In [ ]:
security_knowledge_text = """NIST GUIDELINES: Length matters more than complexity. Encourage passphrasesof 15+ characters over short complex strings. Do not force periodicrotation without evidence of compromise - forced rotation leads topredictable incremented passwords like Password1, Password2. Screen newpasswords against known breached password lists. Encourage MFA (TOTPauthenticator apps or hardware keys) since passwords alone are a weakboundary. Avoid password hints and knowledge-based security questions sincethey are guessable via social engineering.ATTACK PATTERNS: Dictionary attacks try common words and previously leakedpasswords (e.g. rockyou.txt, 14 million real leaked passwords). Brute-forceattacks try every combination - short passwords under 8 characters can becracked in minutes on modern GPUs regardless of symbols used. Credentialstuffing reuses a password leaked from one site against other sites, whichis why unique passwords per account matter. Predictable substitutions likea-to-@ or o-to-0 or appending 123 or an exclamation mark are well known tocracking tools and add little real security. Keyboard-walk patterns likeqwerty or asdfgh are tried very early in cracking attempts.PASSPHRASE CONSTRUCTION: A passphrase of 5-6 random unrelated words drawnfrom a large wordlist can exceed 60 bits of entropy while staying easy tomemorize. Avoid quotes, song lyrics, or famous phrases since these appearin dictionary-attack wordlists built from pop culture. Each word must berandomly selected and unrelated to the others, not a meaningful sentence.Entropy under 28 bits is very weak, 36-59 bits is reasonable for low-valueaccounts, 60-127 bits is strong for most accounts, and 128+ bits suits apassword manager master password."""splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=50)docs = splitter.create_documents([security_knowledge_text])embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")vector_store = FAISS.from_documents(docs, embeddings)retriever = vector_store.as_retriever(search_kwargs={"k": 3})def retrieve_security_guidance(query: str) -> str:    """Retrieve relevant security guidance (NIST rules, attack patterns,    passphrase construction) for a topic. Use this to ground advice instead    of relying on general knowledge."""    results = retriever.invoke(query)    if not results:        return "No relevant guidance found."    return "\n---\n".join(d.page_content for d in results)# quick testprint(retrieve_security_guidance("why is password rotation bad"))

## Tool 4: Strong Password / Passphrase GeneratorUses the LLM to draft options, grounded by what the retriever says about good passphrase construction.

In [ ]:
import secretsimport stringdef generate_strong_password(style: str = "passphrase") -> str:    """Generate a genuinely random strong password or passphrase.    style can be 'passphrase' (memorable, word-based) or 'random'    (dense random characters). Uses Python's cryptographically secure    random generator, not the LLM, so entropy is guaranteed real."""    if style == "random":        alphabet = string.ascii_letters + string.digits + "!@#$%^&*"        pwd = ''.join(secrets.choice(alphabet) for _ in range(16))        return f"random: {pwd}"    else:        wordlist = ["orbit","maple","quartz","ember","velvet","cobalt",                    "harbor","falcon","lantern","granite","willow","copper",                    "marble","thicket","ripple","canyon","ash","drift"]        words = [secrets.choice(wordlist) for _ in range(5)]        pwd = "-".join(words) + str(secrets.randbelow(90) + 10)        return f"passphrase: {pwd}"# quick testprint(generate_strong_password("passphrase"))print(generate_strong_password("random"))

## Step 5: Agent CreationSame pattern as your EDA agent notebook — pass the tool functions straight into `create_agent`.

In [ ]:
tools = [check_password_strength, check_breach_database,         retrieve_security_guidance, generate_strong_password]agent = create_agent(    model=gemini_llm,    tools=tools    )agent

## Step 6: Test the AgentAsk it about a password the same way a user would in the chat UI.

In [ ]:
system_note = (    "You are SentinelAI, a security advisor. Never invent a strength score "    "or breach status yourself - always call the analysis tools and report "    "their real output. Ground explanations using retrieve_security_guidance. "    "Never ask the user to repeat their password back unnecessarily."    )user_password = "Summer2024!"  # example only - never hardcode a real passwordprompt = f"{system_note}\n\nAnalyze this password and explain the risk in " \         f"plain language, then suggest a stronger alternative: {user_password}"response = agent.invoke({'messages': [{'role': 'user', 'content': prompt}]})answer = response["messages"][-1].contentprint(answer)

## Step 7: Report FunctionSame pattern as your `perform_eda_func` report step — wraps the agent call into a single reusable function that returns a full report string.

In [ ]:
def generate_security_report(password: str, agent=agent) -> str:    """Runs the full advisor pipeline for one password and returns a    plain-language report string, ready to display in Streamlit."""    prompt = (        f"{system_note}\n\nGive a full security report for this password: "        f"{password}. Include: 1) strength summary 2) breach status "        f"3) the single biggest weakness 4) one stronger alternative."        )    response = agent.invoke({'messages': [{'role': 'user', 'content': prompt}]})    return response["messages"][-1].contentprint(generate_security_report("qwerty123"))

## Next stepSee `app.py` for the Streamlit deployment of this agent (same structure as your flower classification app — title, input, button, results panel).